In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import rasterio as rio
import rioxarray as rxr
import pcxarray as pcx
from glob import glob
import geopandas as gpd

font_path = '/home/dhester/.fonts/InterVariable.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()

In [ ]:
selected_ids =[
    '095222',
    '139008',
    '149308',
    '210838',
    '037717',
    '042005'
]
samples = gpd.read_parquet('/scratch/dhester/diffusion-examples/s2_naip_pairs_4x_multiyear/samples.par')
selected_samples = samples.loc[samples['id'].isin(selected_ids)]

In [ ]:
def safe_prepare(query_dict):
    try:
        return pcx.query_and_prepare(**query_dict)
    except Exception as e:
        # print(f"Error preparing data for query {query_dict}: {e}")
        return None

all_naip_imgs =[[safe_prepare(dict(
    collections=['naip'],
    geometry=selected_sample.geometry.buffer(320).envelope,
    datetime=str(year),
    all_touched=True,
    crs=selected_samples.crs,
    target_resolution=1.0,
    resampling_method='lanczos',
)) for _, selected_sample in selected_samples.iterrows()] for year in range(2010, 2024)]

In [ ]:
# visualize some samples
fig, axes = plt.subplots(15, len(selected_ids), figsize=(len(selected_ids)*2, 14 * 2), constrained_layout=True)
for i, naip_imgs in enumerate(all_naip_imgs):
    for j, naip_img in enumerate(naip_imgs):
        ax = axes[i, j]
        if naip_img is not None:
            # naip_img = naip_img.rio.clip_box(*selected_samples.iloc[i].geometry.envelope.bounds)
            # naip_img = naip_img / 255.0
            rgb = naip_img.isel(band=[0, 1, 2]).transpose('y', 'x', 'band').values
            rgb = np.clip(rgb / 255.0, 0, 1)
            ax.imshow(rgb)
            ax.set_title(f"{i}", fontsize=12)
        ax.axis('off')

# now plot s2 images at the bottom
for _, selected_sample in selected_samples.iterrows():
    s2_img_path = '../data/s2flow-s2naip/' + selected_sample['input_path']
    try:
        s2_img = rxr.open_rasterio(s2_img_path, masked=True)
    except Exception as e:
        s2_img = None
    if s2_img is not None:
        rgb = s2_img.isel(band=[3, 2, 1]).transpose('y', 'x', 'band').values
        rgb = np.clip(rgb / 10000.0, 0, 1)
        ax.imshow(rgb)
        ax.set_title(f"S2", fontsize=12)
    ax.axis('off')


In [ ]:
naip_img_selected = [
    all_naip_imgs[12][0],
    all_naip_imgs[4][1],
    all_naip_imgs[9][2],
    all_naip_imgs[10][3],
    all_naip_imgs[13][4],
    all_naip_imgs[9][5],
]

In [ ]:


fig, ax = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    naip_img_path = selected_samples.iloc[i]['target_path']
    naip_img_path = '../data/s2flow-s2naip/' + naip_img_path
    naip_img_2_5m = rxr.open_rasterio(naip_img_path) / 10000.0
    
    
    s2_img_path = selected_samples.iloc[i]['input_path']
    s2_img_path = '../data/s2flow-s2naip/' + s2_img_path
    
    s2_img_2_5m = rxr.open_rasterio(s2_img_path).isel(band=[0, 1, 2]) / 10000.0
    s2_img_10m = s2_img_2_5m.rio.reproject(s2_img_2_5m.rio.crs, resolution=(10, 10), resampling=rio.enums.Resampling.lanczos)
    
    ax[0, i].imshow(s2_img_10m.transpose('y', 'x', 'band').values / 10000)
    ax[1, i].imshow(s2_img_2_5m.transpose('y', 'x', 'band').values / 10000)
    ax[2, i].imshow(naip_img_2_5m.transpose('y', 'x', 'band').values / 255.0)
    # ax[3, i].imshow(naip_img.transpose('y', 'x', 'band').values / 255.0)
    
    # ax[0, i].set_title(f"Sample ID: {selected_samples.iloc[i]['id']}", fontsize=10)
    
    for j in range(3):
        ax[j, i].set_xticks([])
        ax[j, i].set_yticks([])


ax[0, 0].set_ylabel('Sentinel-2 10-m', fontsize=14)
ax[1, 0].set_ylabel('Sentinel-2 2.5-m (Upsampled)', fontsize=14)
ax[2, 0].set_ylabel('NAIP 1-m', fontsize=14)

fig.tight_layout()

In [ ]:
pred_imgs = glob('../runs/s2flow_sr_eval_test/sr_outputs/*.tif')

fig, ax = plt.subplots(3, 12, figsize=(32, 8))

gains = [
    1500,
    2500,
    2000,
    4000,
    3500,
    3000,
    2500,
    3000,
    2500,
    3500,
    2500,
    2500,
    
]

for i in range(12):
    sample_id = pred_imgs[i].split('/')[-1].split('.')[0]
    pred_img = rxr.open_rasterio(pred_imgs[i]).isel(band=[0, 1, 2])
    
    naip_sample_path = samples.loc[samples['id'] == sample_id, 'target_path'].iloc[0]
    naip_img = rxr.open_rasterio('../data/s2flow-s2naip/' + naip_sample_path).isel(band=[0, 1, 2])
    
    s2_sample_path = samples.loc[samples['id'] == sample_id, 'input_path'].iloc[0]
    s2_img = rxr.open_rasterio('../data/s2flow-s2naip/' + s2_sample_path).isel(band=[0, 1, 2])
    
    ax[0, i].imshow(s2_img.transpose('y', 'x', 'band').values / gains[i])
    ax[1, i].imshow(pred_img.transpose('y', 'x', 'band').values / gains[i])
    ax[2, i].imshow(naip_img.transpose('y', 'x', 'band').values / gains[i])
    
    for j in range(3):
        ax[j, i].set_xticks([])
        ax[j, i].set_yticks([])

ax[0, 0].set_ylabel('Sentinel-2 Input', fontsize=14)
ax[1, 0].set_ylabel('Super-Resolved Image', fontsize=14)
ax[2, 0].set_ylabel('NAIP 2.5-m Reference', fontsize=14)

fig.tight_layout()
    

In [ ]:
pred_imgs = glob('../runs/s2flow_sr_eval_test/sr_outputs/*.tif')

fig, ax = plt.subplots(3, 12, figsize=(32, 8))

gains = [
    2500,
    3000,
    2500,
    3500,
    3500,
    1500,
    3500,
    2500,
    4000,
    3000,
    2000,
    2500,
    
]

for i in range(12):
    sample_id = pred_imgs[i+12].split('/')[-1].split('.')[0]
    pred_img = rxr.open_rasterio(pred_imgs[i+12]).isel(band=[0, 1, 2])
    
    naip_sample_path = samples.loc[samples['id'] == sample_id, 'target_path'].iloc[0]
    naip_img = rxr.open_rasterio('../data/s2flow-s2naip/' + naip_sample_path).isel(band=[0, 1, 2])
    
    s2_sample_path = samples.loc[samples['id'] == sample_id, 'input_path'].iloc[0]
    s2_img = rxr.open_rasterio('../data/s2flow-s2naip/' + s2_sample_path).isel(band=[0, 1, 2])
    
    ax[0, i].imshow(s2_img.transpose('y', 'x', 'band').values / gains[i])
    ax[1, i].imshow(pred_img.transpose('y', 'x', 'band').values / gains[i])
    ax[2, i].imshow(naip_img.transpose('y', 'x', 'band').values / gains[i])
    
    for j in range(3):
        ax[j, i].set_xticks([])
        ax[j, i].set_yticks([])

ax[0, 0].set_ylabel('Sentinel-2 Input', fontsize=14)
ax[1, 0].set_ylabel('Super-Resolved Image', fontsize=14)
ax[2, 0].set_ylabel('NAIP 2.5-m Reference', fontsize=14)

fig.tight_layout()

In [ ]:
pred_imgs = glob('../runs/s2flow_sr_eval_test/sr_outputs/*.tif')

fig, ax = plt.subplots(3, 12, figsize=(32, 8))

gains = [
    3000,
    3000,
    2500,
    1500,
    3500,
    2500,
    2500,
    2500,
    4000,
    2500,
    2500,
    2500,
    
]

for i in range(12):
    sample_id = pred_imgs[i+24].split('/')[-1].split('.')[0]
    pred_img = rxr.open_rasterio(pred_imgs[i+24]).isel(band=[0, 1, 2])
    
    naip_sample_path = samples.loc[samples['id'] == sample_id, 'target_path'].iloc[0]
    naip_img = rxr.open_rasterio('../data/s2flow-s2naip/' + naip_sample_path).isel(band=[0, 1, 2])
    
    s2_sample_path = samples.loc[samples['id'] == sample_id, 'input_path'].iloc[0]
    s2_img = rxr.open_rasterio('../data/s2flow-s2naip/' + s2_sample_path).isel(band=[0, 1, 2])
    
    ax[0, i].imshow(s2_img.transpose('y', 'x', 'band').values / gains[i])
    ax[1, i].imshow(pred_img.transpose('y', 'x', 'band').values / gains[i])
    ax[2, i].imshow(naip_img.transpose('y', 'x', 'band').values / gains[i])
    
    for j in range(3):
        ax[j, i].set_xticks([])
        ax[j, i].set_yticks([])

ax[0, 0].set_ylabel('Sentinel-2 Input', fontsize=14)
ax[1, 0].set_ylabel('Super-Resolved Image', fontsize=14)
ax[2, 0].set_ylabel('NAIP 2.5-m Reference', fontsize=14)

fig.tight_layout()

In [ ]:
# input, DDIM, DDPM, FM (Euler n_steps=1), FM (midpoint n_steps=1), RealESRGAN, NAIP

from pathlib import Path
from s2flow.engine.sampling import EulerSampler, MidpointSampler, DDPMSampler, DDIMSampler, GANSampler
from s2flow.models import UNetTensorWrapper, RRDBNet
import torch
import yaml
import matplotlib.pyplot as plt
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
import numpy as np

fm_model_config = yaml.safe_load(open('../logs/s2flow_sr_train/config.yaml'))

fm_model = UNetTensorWrapper(fm_model_config['sr_model'])
fm_model.load_state_dict(torch.load('../runs/s2flow_sr_train/model.pt', map_location='cpu'), strict=False)
fm_model.to('cuda')

euler_cfg = {
    'hyperparameters': {
        'use_amp': True,
    },
    'sampling': {
        'num_steps': 1,
    }
}

euler_1 = EulerSampler(euler_cfg, model=fm_model)
midpoint_20 = MidpointSampler({'hyperparameters': {'use_amp': True}, 'sampling': {'num_steps': 20}}, model=fm_model)

gan_model_config = yaml.safe_load(open('../logs/s2flow_realesrgan/config.yaml'))
gan_model = RRDBNet(gan_model_config['sr_model'])
gan_model.load_state_dict(torch.load('../runs/s2flow_realesrgan/model.pt', map_location='cpu'), strict=False)
gan_model.to('cuda')
gan_sampler = GANSampler({'hyperparameters': {'use_amp': True}}, model=gan_model)

ddpm_model = UNetTensorWrapper(fm_model_config['sr_model'])
ddpm_model.load_state_dict(torch.load('../runs/s2flow_sr_train_ddpm/model.pt', map_location='cpu'), strict=False)
ddpm_model.to('cuda')
ddpm_sampler = DDPMSampler({'hyperparameters': {'use_amp': True}, 'sampling': {'num_steps': 100}}, model=ddpm_model)
ddim_sampler = DDIMSampler({'hyperparameters': {'use_amp': True}, 'sampling': {'num_steps': 100}}, model=ddpm_model)

# samples = gpd.read_parquet('../data/s2flow-s2naip/samples.par')
base_path = Path('/scratch/dhester/diffusion-examples/s2_naip_pairs_4x_multiyear/')
samples = gpd.read_parquet(base_path / 'samples.par')
selected_samples = samples.loc[samples['id'].isin(selected_ids)]

naip_img = []
s2_img = []
for i, sample_id in enumerate(selected_ids):
    
    naip_img.append(rxr.open_rasterio(base_path / selected_samples.loc[selected_samples['id'] == sample_id, 'target_path'].iloc[0]))
    s2_img.append(rxr.open_rasterio(base_path / selected_samples.loc[selected_samples['id'] == sample_id, 'input_path'].iloc[0]))
    
naip_img = np.stack(naip_img)
s2_img = np.stack(s2_img)

s2_img_scaled = (s2_img / 10000.0) * 2.0 - 1.0

s2_img_scaled_tensor = torch.from_numpy(s2_img_scaled).float().to('cuda')

s2_img_scaled_tensor.shape

with torch.no_grad():
    euler_1_img_tensor = euler_1.sample(s2_img_scaled_tensor)
    midpoint_20_img_tensor = midpoint_20.sample(s2_img_scaled_tensor)
    gan_img_tensor = gan_sampler.sample(s2_img_scaled_tensor)
    ddpm_img_tensor = ddpm_sampler.sample(s2_img_scaled_tensor)
    ddim_img_tensor = ddim_sampler.sample(s2_img_scaled_tensor)


In [ ]:
gains = [
    5,
    5,
    3.5,
    3.5,
    3.5,
    4,
]

# zoom indices
zoom_indices = [
    # top left
    (slice(0, 128), slice(0, 128)),
    # top right
    (slice(0, 128), slice(-128, 256)),
    # center
    (slice(64, 192), slice(64, 192)),
    # top left
    (slice(0, 128), slice(0, 128)),
    # bottom right
    (slice(-128, 256), slice(-128, 256)),
]

s2_img_down_tensor = torch.nn.functional.interpolate(
    s2_img_scaled_tensor,
    scale_factor=0.25,
    mode='bicubic',
    align_corners=False,
)


fig, ax = plt.subplots(5, 8, figsize=(16, 10))
for i in range(5):
    
    s2_img_i = xr.DataArray(s2_img_scaled_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    s2_img_down_i = xr.DataArray(s2_img_down_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    naip_img_i = xr.DataArray(naip_img[i], dims=['band', 'y', 'x'])
    
    euler_1_img = xr.DataArray(euler_1_img_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    gan_img = xr.DataArray(gan_img_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    midpoint_20_img = xr.DataArray(midpoint_20_img_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    ddpm_img = xr.DataArray(ddpm_img_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    ddim_img = xr.DataArray(ddim_img_tensor[i].cpu().numpy(), dims=['band', 'y', 'x'])
    
    # scale to [0, 1]
    s2_img_i = (s2_img_i + 1.0) / 2.0
    s2_img_down_i = (s2_img_down_i + 1.0) / 2.0
    euler_1_img = (euler_1_img + 1.0) / 2.0
    gan_img = (gan_img + 1.0) / 2.0
    midpoint_20_img = (midpoint_20_img + 1.0) / 2.0
    ddpm_img = (ddpm_img + 1.0) / 2.0
    ddim_img = (ddim_img + 1.0) / 2.0
    naip_img_i = naip_img_i / 10000.0
    
    down_zoom_indices = (slice(zoom_indices[i][0].start // 4, zoom_indices[i][0].stop // 4),
                         slice(zoom_indices[i][1].start // 4, zoom_indices[i][1].stop // 4))
    ax[i, 0].imshow(s2_img_down_i.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[down_zoom_indices] * gains[i])
    ax[i, 1].imshow(s2_img_i.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 2].imshow(euler_1_img.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 3].imshow(ddim_img.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 4].imshow(ddpm_img.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 5].imshow(gan_img.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 6].imshow(midpoint_20_img.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])
    ax[i, 7].imshow(naip_img_i.transpose('y', 'x', 'band').isel(band=[0, 1, 2])[zoom_indices[i]] * gains[i])

    for j in range(8):
        ax[i, j].set_xticks([])
        ax[i, j].set_yticks([])
        # remove border
        ax[i, j].spines[:].set_visible(False)
        
        

fontsize = 17
ax[0, 0].set_title('Sentinel-2 Input', fontsize=fontsize)
ax[0, 1].set_title('Lanczos', fontsize=fontsize)
ax[0, 2].set_title('Euler ($T=1$)', fontsize=fontsize)
ax[0, 3].set_title('DDIM ($T=100$)', fontsize=fontsize)
ax[0, 4].set_title('DDPM ($T=100$)', fontsize=fontsize)
ax[0, 5].set_title('Real-ESRGAN', fontsize=fontsize)
ax[0, 6].set_title('Midpoint ($T=20$)', fontsize=fontsize)
ax[0, 7].set_title('NAIP Reference', fontsize=fontsize)

# label rows
ax[0, 0].set_ylabel('(a)', fontsize=fontsize, rotation=0, labelpad=15)
ax[1, 0].set_ylabel('(b)', fontsize=fontsize, rotation=0, labelpad=15)
ax[2, 0].set_ylabel('(c)', fontsize=fontsize, rotation=0, labelpad=15)
ax[3, 0].set_ylabel('(d)', fontsize=fontsize, rotation=0, labelpad=15)
ax[4, 0].set_ylabel('(e)', fontsize=fontsize, rotation=0, labelpad=15)


fig.tight_layout()

import os
os.makedirs('../figures/s2flow_sr_model_comparison', exist_ok=True)
plt.savefig('../figures/s2flow_sr_model_comparison/s2flow_sr_model_comparison.png', dpi=300)
plt.savefig('../figures/s2flow_sr_model_comparison/s2flow_sr_model_comparison.pdf', dpi=300)


In [ ]:

# not plot a bunch of selected preds
# first, plot several grids of ground truth validation samples to select good tiles for visualization

val_samples = samples.loc[samples['split'] == 'val'][::-1]
# val_samples

# for i in range(10):
    
#     fig, ax = plt.subplots(5, 5, figsize=(20, 20))
    
#     for j, (_, val_sample) in enumerate(val_samples[i*25:(i+1)*25].iterrows()):
#         naip_img = rxr.open_rasterio(base_path / val_sample['target_path']).isel(band=[0, 1, 2])
#         rgb = naip_img.transpose('y', 'x', 'band').values
#         rgb = rgb / 2500
#         ax[j // 5, j % 5].imshow(rgb)
#         ax[j // 5, j % 5].set_title(f"Sample ID: {val_sample['id']}", fontsize=10)
#         ax[j // 5, j % 5].set_xticks([])
#         ax[j // 5, j % 5].set_yticks([])

selected_samples = [
    248439,
    247459,
    246579,
    245419,
    245039,
    240259,
    243539,
    240279,
    240919,
]

base_path = Path('/scratch/dhester/diffusion-examples/s2_naip_pairs_4x_multiyear/')
sr_out_path = Path('../runs/s2flow_sr_eval_steps/s2flow_sr_eval_steps_20/sr_outputs/')

selected_samples = samples.loc[samples['id'].isin([str(s) for s in selected_samples])]

k_indices = {
    1: [0, 1, 2], # RGB
    0: [3, 0, 1], # Color Infrared
}

gain = 2.8
inset_padding = 0.01


fig, ax = plt.subplots(3, 3, figsize=(36, 24))

title_font_size = 32
y_axis_font_size = 28

middle_only = True

for i in range(3):
    
    for j in range(3):
        
        # create 6 axes inset (2 rows, 3 cols)
        
        inset_axes = [[]]
        
        for k in range(2):
            
            inset_axes.append([])
            
            for l in range(3):
                
                # inset_axes[k].append(ax[i, j].inset_axes([l * (1/3), k * (1/2), 1/3, 1/2]))
                inset_axes[k].append(ax[i, j].inset_axes([
                    l * (1/3) + inset_padding,
                    k * (1/2) + inset_padding,
                    (1/3) - 2 * inset_padding,
                    (1/2) - 2 * inset_padding,
                ]))
                inset_axes[k][l].set_xticks([])
                inset_axes[k][l].set_yticks([])
                inset_axes[k][l].spines[:].set_visible(False)
                
                if l == 0: # plot Sentinel-2
                    img_path = base_path / selected_samples.iloc[i + j * 3]['input_path']
                    img = rxr.open_rasterio(img_path)
                
                if l == 1: # plot super-resolved
                    img_path = sr_out_path / f"{selected_samples.iloc[i + j * 3]['id']}.tif"
                    img = rxr.open_rasterio(img_path)
                
                if l == 2: # plot NAIP
                    img_path = base_path / selected_samples.iloc[i + j * 3]['target_path']
                    img = rxr.open_rasterio(img_path)
                
                img = img.isel(band=k_indices[k])
                if l == 0:
                    img = img.rio.reproject(img.rio.crs, resolution=(10, 10), resampling=rio.enums.Resampling.lanczos)
                if middle_only:
                    if l == 0:
                        img = img.isel(x=slice(16, -16), y=slice(16, -16))
                    else:
                        img = img.isel(x=slice(64, -64), y=slice(64, -64))
                img = img.transpose('y', 'x', 'band').values * gain
                img = np.clip(img / 10000, 0, 1)
                inset_axes[k][l].imshow(img)
                
                if i == 0 and k == 1:
                    if l == 0:
                        inset_axes[k][l].set_title('Sentinel-2 Input', fontsize=title_font_size, y=1.03)
                    elif l == 1:
                        inset_axes[k][l].set_title('Super-Resolved', fontsize=title_font_size, y=1.03)
                    else:
                        inset_axes[k][l].set_title('NAIP Reference', fontsize=title_font_size, y=1.03)
                
                if j == 0 and l == 0:
                    if k == 0:
                        inset_axes[k][l].set_ylabel('Color Infrared', fontsize=y_axis_font_size)
                    else:
                        inset_axes[k][l].set_ylabel('Visible RGB', fontsize=y_axis_font_size)
        
        # remove borders and ticks from main axes
        ax[i, j].set_xticks([])
        ax[i, j].set_yticks([])
        ax[i, j].spines[:].set_visible(False)


fig.tight_layout()

# add some spacing between subplots
fig.subplots_adjust(wspace=0.1, hspace=0.1)


out_path = '../figures/sr_results_grid/'
os.makedirs(out_path, exist_ok=True)
for file_ext in ['png', 'pdf']:
    plt.savefig(f'{out_path}/s2flow_sr_results_grid.{file_ext}', dpi=300)

In [ ]:
#